In [1]:
# std lib imports
import hashlib

# 3 party import
import numpy as np
import torch
import matplotlib.pyplot as plt

# projekt imports
from RadarDataGen.Data_Generator.pseudo_radar_points import pseudo_radar_points
from RadarDataGen.Discretizer.radar_discretizer import RadarDiscretizer
from RadarDataGen.Data_Generator.generator import PseudoRadarGridGenerator, StreamingRadarDataset, RadarDataset, worker_init_fn

## Pseudo Radar Datasets
In this notebook we will show what the pseudo Radar datasets are and how one can work with them 

The Parameters we will use in this notebook

In [ ]:
# discretizer params
discretizer_params = {
    "grid_size": 64,
    "x_min": -1,
    "x_max": 1,
    "y_min": -1,
    "y_max": 1,
    "valid_indicator": 1.0
  }

# generated point cloud params
point_cloud_params = {
    "lambda_lines_2d": 5,
    "lambda_points_line_2d": 20,
    "lambda_clutter": 50
}


In [3]:
# generate a Diskretizer 
discretizer = RadarDiscretizer(**discretizer_params)

# generate a Pseudo Radar Grid generator that generates points and then diskretizes them using the discretizer
generator = PseudoRadarGridGenerator(params=point_cloud_params, discretizer_params=discretizer_params)

# settings
batch_size = 2 

In [4]:
def hash_tensor(array : np.array):
    array_bytes = array.tobytes()
    return hashlib.sha256(array_bytes).hexdigest()

# Create a streaming radar dataset and use it with a Torch.Dataloader
To create a pseudo torch radar dataset we need to create a discretizer and a sampler.

In [ ]:
dataset = StreamingRadarDataset(
    sampler=generator,
    dtype=torch.float16,
    base_seed=42,
)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    worker_init_fn=worker_init_fn,
)

stream_dataset_loader = iter(loader) 
callable_loader = lambda: next(stream_dataset_loader)

### Use it as a callable

In [ ]:
num_rows, num_cols = 10, 4


fig, ax = plt.subplots(num_rows, num_cols, figsize=(num_rows * 10, num_cols * 30), tight_layout=True)
ax = ax.ravel()

num_samples = num_rows * num_cols 
ax_index = 0

for i in range(num_samples // 2):
    batch = callable_loader()

    for idx, grid in enumerate(batch):
        image = discretizer.grid_to_image(grid.to("cpu").numpy().astype(np.float32).swapaxes(0, -1), swap_xy=True, invert_rows=True, invert_columns=False)
        ax[ax_index].matshow(image);
        ax_index += 1

    if ax_index >= num_samples - 1:
        break

fig.show()

### Use it as an iterable

In [ ]:
num_rows, num_cols = 10, 4
fig, ax = plt.subplots(num_rows, num_cols, figsize=(num_rows * 10, num_cols * 30), tight_layout=True)
ax = ax.ravel()

num_samples = num_rows * num_cols 
ax_index = 0

for batch in loader:
    for idx, grid in enumerate(batch):
        image = discretizer.grid_to_image(grid.to("cpu").numpy().astype(np.float32).swapaxes(0, -1), swap_xy=True, invert_rows=True, invert_columns=False)
        ax[ax_index].matshow(image);
        ax_index += 1

    if ax_index >= num_samples - 1:
        break

fig.show()

## As we wanted to have a streaming Dataset see if we dublicate Samples 

In [ ]:
hash_list = []
num_samples = 1000
index = 0

for batch in dataset:
    for grid in batch:
        array_hash = hash_tensor(grid.to("cpu").numpy())
        if array_hash in hash_list:
            raise ValueError(f"Duplicate sample found at index {index} with hash {array_hash}")
        hash_list.append(array_hash)
        index += 1

    if index >= num_samples - 1:
        break

print("No duplicate samples found.")

# Create a radar dataset with fixed Size and use it with a Torch.Dataloader


In [5]:
num_dataset_samples = 5

In [6]:
dataset = RadarDataset(
    sampler=generator,
    dtype=torch.float16,
    base_seed=42,
    num_samples=num_dataset_samples
)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=1,
    worker_init_fn=worker_init_fn,
)

stream_dataset_loader = iter(loader) 
callable_loader = lambda: next(stream_dataset_loader)

### Use it as an callable

In [ ]:
num_rows, num_cols = 10, 4


fig, ax = plt.subplots(num_rows, num_cols, figsize=(num_rows * 10, num_cols * 30), tight_layout=True)
ax = ax.ravel()

num_samples = num_rows * num_cols 
ax_index = 0

for i in range(num_samples // 2):
    batch = callable_loader()

    for idx, grid in enumerate(batch):
        image = discretizer.grid_to_image(grid.to("cpu").numpy().astype(np.float32).swapaxes(0, -1), swap_xy=True, invert_rows=True, invert_columns=False)
        ax[ax_index].matshow(image);
        ax_index += 1

    if ax_index >= num_samples - 1:
        break

fig.show()

### Use it as an iterable

In [9]:
num_rows, num_cols = 10, 4
fig, ax = plt.subplots(num_rows, num_cols, figsize=(num_rows * 10, num_cols * 30), tight_layout=True)
ax = ax.ravel()

num_samples = num_rows * num_cols 
ax_index = 0

for batch in loader:
    for idx, grid in enumerate(batch):
        image = discretizer.grid_to_image(grid.to("cpu").numpy().astype(np.float32).swapaxes(0, -1), swap_xy=True, invert_rows=True, invert_columns=False)
        ax[ax_index].matshow(image);
        ax_index += 1

    if ax_index >= num_samples - 1:
        break

fig.show()

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.56396484..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.7578125..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.7578125..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.77197266..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.77197266..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.3291016..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.56396484..1.0].
Clipping input data to 

### See if we have a fixed dataset with num_dataset_samples

In [10]:
hash_list = []
num_samples = 10000
index = 0

# iterate over the first half of the num samples
for batch in loader:
    for grid in batch:
        array_hash = hash_tensor(grid.to("cpu").numpy())
        hash_list.append(array_hash)    
        
        index += 1

    if index >= (num_samples // 2)- 1:
        break

for i in range(num_samples // batch_size // 2):
    batch = callable_loader()

    for grid in batch:
        array_hash = hash_tensor(grid.to("cpu").numpy())
        hash_list.append(array_hash)    


print(f"Unique Samples Generated: {len(set(hash_list))}, should be {num_dataset_samples} Unique Samples")

Unique Samples Generated: 5, should be 5 Unique Samples
